# Plantilla: clasificación binaria (agnóstica al dataset)

Notebook **base** para predecir una variable con **exactamente dos clases** (sí/no, 0/1, spam/ham…). No está ligado a un dataset concreto.

## Pasos principales (ejecutar en orden)

| Paso | Sección | Qué haces |
|------|---------|-----------|
| **0** | Helpers | Imports y funciones compartidas de preprocesado y modelos. |
| **1** | Explorar CSV | `PREVIEW_PATH`, `PREVIEW_SEP` — columnas, tipos, faltantes, conteo del target. |
| **2** | CONFIG | `DATA_PATH`, `TARGET_COL`, `DROP_COLS`; opcional listas de columnas; `build_models()`. |
| **3** | Carga | Leer el CSV. |
| **4** | Calidad de datos | Balance de clases y faltantes. |
| **5** | Visualización | Barras de frecuencia por clase. |
| **6** | Split | **X**, **y**; `train_test_split` **estratificado** (`stratify=y`). |
| **7** | Preprocesado | Numéricas → imputer + escalar; categóricas (texto) → imputer + one-hot. |
| **8** | Comparar modelos | Benchmark con la misma transformación; orden por **F1** (weighted). |
| **9** | Mejor modelo | `classification_report` y matriz de confusión en test. |

Mismo criterio anti-*leakage*: preprocesado dentro del `Pipeline`. Categóricas en texto (`object`, `category`, `bool`, `string`) pasan por one-hot.

### Primera vez con tu CSV

1. Copia el archivo a `data/`.
2. Paso **1** (explorar) → ajusta separador y revisa que el target tenga **dos** clases.
3. Paso **2** (CONFIG) con los mismos path/separador.
4. Pasos **3**–**9** en orden.

**Ejemplos ya resueltos:** `02-clasificacion-binaria-breast-cancer.ipynb` · `02-clasificacion-binaria-bank-marketing.ipynb`

> Ejecuta Jupyter desde `07.b-ejemplos-supervisados/`.



In [ ]:
# =============================================================================
# Helpers — funciones reutilizables (misma lógica en todo el benchmark)
# =============================================================================
import importlib  # importar xgboost/catboost solo si están instalados
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer  # pipeline distinto por tipo de columna
from sklearn.impute import SimpleImputer  # rellenar NaN antes de escalar/codificar
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline  # encadena: preprocesado → modelo
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")


def infer_feature_columns(df, target_col, drop_cols, feature_cols):
    """Lista de columnas predictoras (X).

    - Si FEATURE_COLS no es None: usa esa lista explícita.
    - Si no: todas las columnas excepto TARGET_COL y DROP_COLS (ids, leakage…).
    """
    if feature_cols is not None:
        return list(feature_cols)
    exclude = {target_col, *drop_cols}
    return [c for c in df.columns if c not in exclude]


def infer_column_types(X, numeric_cols=None, categorical_cols=None):
    """Separa columnas para ColumnTransformer.

    Por defecto: int/float → numéricas; object/category/bool/string → categóricas.
    Si solo defines NUMERIC_COLS, las categóricas siguen infiriéndose (y viceversa).
    Para control total, define ambas listas en CONFIG.
    """
    if numeric_cols is None:
        numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    if categorical_cols is None:
        categorical_cols = X.select_dtypes(
            include=["object", "category", "bool", "string"]
        ).columns.tolist()
    return list(numeric_cols), list(categorical_cols)


def build_preprocess(numeric_cols, categorical_cols):
    """Preprocesador único compartido por todos los modelos del benchmark.

    Numéricas: imputar (mediana) → StandardScaler.
    Categóricas (texto): imputar (moda) → OneHotEncoder.
    El .fit() ocurre dentro de pipe.fit(X_train) — en test solo .transform() (sin leakage).
    Columnas de X no listadas aquí se descartan (remainder='drop').
    """
    transformers = []

    if numeric_cols:
        transformers.append(
            (
                "num",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_cols,
            )
        )

    if categorical_cols:
        transformers.append(
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "encoder",
                            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                        ),
                    ]
                ),
                categorical_cols,
            )
        )

    if not transformers:
        raise ValueError("No hay columnas numéricas ni categóricas para preprocesar.")
    return ColumnTransformer(transformers, remainder="drop")


def _optional_estimator(module_name, class_name, **kwargs):
    """Instancia XGBoost/CatBoost si el paquete está instalado; si no, None (se omite del benchmark)."""
    try:
        module = importlib.import_module(module_name)
        cls = getattr(module, class_name)
        return cls(**kwargs)
    except ImportError:
        print(f"  [aviso] {module_name}.{class_name} no instalado (pip install {module_name})")
        return None



## 1. Explorar el CSV (antes de CONFIG)

Pon la ruta de **tu** archivo. Si todo aparece en una columna, cambia `PREVIEW_SEP`.

In [ ]:
# --- Paso 1: explorar SIN tocar CONFIG todavía ---
PREVIEW_PATH = "data/mi_dataset.csv"  # ruta a tu CSV
PREVIEW_SEP = ","  # separador: ","  |  ";"  |  "\t"

# Carga provisional solo para inspeccionar estructura
df_preview = pd.read_csv(PREVIEW_PATH, sep=PREVIEW_SEP)

print(f"Filas: {len(df_preview):,}  |  Columnas: {len(df_preview.columns)}")
print("\n--- Nombres de columnas (índice : nombre) ---")
for i, col in enumerate(df_preview.columns):
    print(f"  {i:2d}: {col!r}")

print("\n--- Tipos de datos (dtypes) ---")
print(df_preview.dtypes)

print("\n--- Primeras filas ---")
display(df_preview.head())

# Faltantes: el pipeline imputará después; aquí solo diagnosticamos
print("\n--- Valores faltantes por columna ---")
missing = df_preview.isna().sum()
if missing.any():
    display(missing[missing > 0].to_frame("nulos"))
else:
    print("No hay valores faltantes.")

# Ayuda para rellenar NUMERIC_COLS / CATEGORICAL_COLS en CONFIG
_num = df_preview.select_dtypes(include=[np.number]).columns.tolist()
_cat = df_preview.select_dtypes(include=["object", "category", "bool", "string"]).columns.tolist()
print("\n--- Sugerencia automática de tipos ---")
print("Numéricas (int/float):", _num)
print("Categóricas (object/category/bool/string):", _cat)
print(
    "\n>>> Siguiente: en CONFIG pon DATA_PATH, CSV_SEP iguales y elige TARGET_COL "
    "(columna con exactamente **dos** clases)."
)



## 2. CONFIG — adaptar a tu dataset

Copia los valores de la exploración. **Solo esta sección** cambia entre proyectos.


In [ ]:
# ========== Paso 2: CONFIG — único bloque que cambia entre datasets ==========
DATA_PATH = "data/mi_dataset.csv"  # mismo path que PREVIEW_PATH
CSV_SEP = ","  # mismo separador que PREVIEW_SEP

TARGET_COL = "nombre_columna_clase"  # exactamente 2 clases (0/1, yes/no…)

# Columnas que no deben usarse como features (ids, texto libre, leakage)
DROP_COLS = []  # ej. ["id", "car_name"]

# None = automático; o listas explícitas si la inferencia falla
FEATURE_COLS = None
NUMERIC_COLS = None
CATEGORICAL_COLS = None

TEST_SIZE = 0.2  # 20 % test, 80 % train
RANDOM_STATE = 42  # reproducibilidad del split y modelos
METRIC_PRINCIPAL = "f1"  # columna para ordenar la tabla (mayor = mejor en clasificación binaria (F1 weighted))


def build_models():
    """Diccionario nombre → estimador. Comenta líneas para excluir modelos del benchmark."""
    from sklearn.ensemble import (
        GradientBoostingClassifier,
        HistGradientBoostingClassifier,
        RandomForestClassifier,
    )
    from sklearn.linear_model import LogisticRegression

    models = {
        "LogisticRegression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        "RandomForest": RandomForestClassifier(
            n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
        ),
        "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "HistGradientBoosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
    }
    xgb = _optional_estimator(
        "xgboost", "XGBClassifier",
        random_state=RANDOM_STATE, verbosity=0, n_estimators=100,
        eval_metric="logloss", n_jobs=-1,
    )
    if xgb is not None:
        models["XGBoost"] = xgb
    cat = _optional_estimator(
        "catboost", "CatBoostClassifier",
        random_state=RANDOM_STATE, verbose=False, iterations=100,
        allow_writing_files=False,
    )
    if cat is not None:
        models["CatBoost"] = cat
    return models


# Se evalúa una sola vez; el benchmark usa este diccionario
MODELS = build_models()



## 3. Carga de datos


In [ ]:
# --- Paso 3: carga definitiva con los parámetros de CONFIG ---
df = pd.read_csv(DATA_PATH, sep=CSV_SEP)
print("Shape:", df.shape)
df.head()



## 4. Calidad de datos


In [ ]:
# --- Paso 4: balance de clases y faltantes ---
print(df[TARGET_COL].value_counts())  # ¿hay desbalance entre las clases?
print("\nFaltantes:")
print(df.isna().sum().pipe(lambda s: s[s > 0] if s.any() else "Sin faltantes"))



## 5. Visualización rápida


In [ ]:
# --- Paso 5: conteo por clase ---
fig, ax = plt.subplots(figsize=(6, 4))
df[TARGET_COL].value_counts().plot(kind="bar", ax=ax)
ax.set_title("Distribución de clases")
ax.set_xlabel(TARGET_COL)
plt.tight_layout()
plt.show()



## 6. X / y y split (estratificado)


In [ ]:
# --- Paso 6: separar features (X), target (y) y dividir train / test ---
feature_cols = infer_feature_columns(df, TARGET_COL, DROP_COLS, FEATURE_COLS)
X = df[feature_cols]
y = df[TARGET_COL]

# Clasificación de columnas para el ColumnTransformer
numeric_cols, categorical_cols = infer_column_types(X, NUMERIC_COLS, CATEGORICAL_COLS)
print("Numéricas:", len(numeric_cols), "| Categóricas:", len(categorical_cols))

# stratify=y mantiene la proporción de clases en train y test (solo tiene sentido con clases discretas)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)



## 7. Preprocesado


In [ ]:
# --- Paso 7: definir el preprocesador (mismo objeto para todos los modelos) ---
preprocess = build_preprocess(numeric_cols, categorical_cols)
preprocess  # muestra la estructura: ramas num y cat



## 8. Comparar modelos


In [ ]:
# --- Paso 8: métricas y benchmark de modelos ---

def classification_metrics(y_true, y_pred):
    """Accuracy, precision, recall y F1 (weighted) en test."""
    from sklearn.metrics import (
        accuracy_score,
        f1_score,
        precision_score,
        recall_score,
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


def evaluate_models(models, preprocess, X_train, X_test, y_train, y_test):
    """Mismo preprocesado y mismo split para comparar algoritmos de forma justa."""
    rows = []
    for name, estimator in models.items():
        pipe = Pipeline([("preprocess", preprocess), ("model", estimator)])
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        rows.append({"modelo": name, **classification_metrics(y_test, y_pred)})
    return pd.DataFrame(rows).sort_values(METRIC_PRINCIPAL, ascending=False)


results = evaluate_models(MODELS, preprocess, X_train, X_test, y_train, y_test)
display(results.round(4))

ax = results.plot(x="modelo", y=METRIC_PRINCIPAL, kind="barh", legend=False, figsize=(8, 5))
ax.set_xlabel(f"{METRIC_PRINCIPAL} (test)")
ax.set_title("Comparación de modelos — clasificación binaria")
plt.tight_layout()
plt.show()



## 9. Detalle del mejor modelo


In [ ]:
# --- Paso 9: reporte y matriz de confusión del mejor modelo ---
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

best_name = results.iloc[0]["modelo"]
print(f"Mejor modelo (test): {best_name}\n")

best_pipe = Pipeline([("preprocess", preprocess), ("model", MODELS[best_name])])
best_pipe.fit(X_train, y_train)
y_pred = best_pipe.predict(X_test)

# Métricas por clase (precision/recall/f1 de cada etiqueta)
print(classification_report(y_test, y_pred))

# Matriz: filas = clase real, columnas = clase predicha
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title(f"Matriz de confusión — {best_name}")
plt.tight_layout()
plt.show()



## Checklist: nuevo dataset (binaria)

1. CSV en `data/` → explorar (sección 1) → CONFIG (sección 2).
2. `TARGET_COL` debe tener **solo dos clases** (o mapea a 0/1).
3. `DROP_COLS`: ids, columnas con *leakage* (información del futuro).
4. Comprueba balance de clases antes del split estratificado.
5. Métrica principal habitual: **F1** (especialmente si hay desbalance).
